# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/afreensumai64/ML-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [29]:
# W07 setup — build the action queue from the validated model

import os
import pandas as pd
import numpy as np

os.makedirs("work/outputs", exist_ok=True)

print("Output directory: READY")

Output directory: READY


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Ranked Action Queue

The action queue ranks pages using the model score and assigns a human-readable reason code.

The purpose is to help a content team decide which pages to inspect first.

The action labels are:

- `review_refresh` — inspect the page for freshness, relevance, and content-quality opportunities.
- `review_visibility` — inspect search visibility and query/page alignment.
- `monitor` — retain the page for monitoring rather than prioritizing immediate review.

The ranking is a decision-support output. A high score does not automatically mean that a page should be changed.

In [30]:
# ML-10 — Content Action Playbook
# Section 1 — Build a reproducible ranked action queue

import os
import pandas as pd
import numpy as np
import duckdb

from google.colab import userdata

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


# ============================================================
# 1. CONNECT TO FLYRANK DATASET
# ============================================================

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN was not found in Colab Secrets. "
        "Make sure the secret is named HF_TOKEN and notebook access is ON."
    )

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf_secret
    (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    )
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FEB = (
    f"{REL}/fact_content_daily_performance/"
    f"month=2026-02/*.parquet"
)

MAR = (
    f"{REL}/fact_content_daily_performance/"
    f"month=2026-03/*.parquet"
)

DIM_CONTENT = f"{REL}/dim_content.parquet"

print("DuckDB connection: READY")
print("FlyRank warehouse: READY")


# ============================================================
# 2. BUILD FEBRUARY DECISION-TIME FEATURES
# ============================================================

FEB_FEATURES = con.sql(
    f"""
    WITH feb AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS gsc_impressions,
            SUM(gsc_clicks) AS gsc_clicks

        FROM read_parquet('{FEB}')

        WHERE gsc_data_available IS TRUE

        GROUP BY
            client_hash_id,
            content_hash_id
    ),

    content AS (
        SELECT
            client_hash_id,
            content_hash_id,
            content_created_date

        FROM read_parquet('{DIM_CONTENT}')
    )

    SELECT
        f.client_hash_id,
        f.content_hash_id,
        f.gsc_impressions,
        f.gsc_clicks,

        DATE_DIFF(
            'day',
            c.content_created_date,
            DATE '2026-02-28'
        ) AS content_age_days

    FROM feb f

    INNER JOIN content c
        ON f.client_hash_id = c.client_hash_id
       AND f.content_hash_id = c.content_hash_id

    WHERE c.content_created_date IS NOT NULL
    """
).df()

print(
    "February feature rows:",
    len(FEB_FEATURES)
)


# ============================================================
# 3. BUILD MARCH OBSERVED OUTCOME
# ============================================================

MAR_OUTCOME = con.sql(
    f"""
    SELECT
        client_hash_id,
        content_hash_id,

        CASE
            WHEN SUM(gsc_clicks) = 0 THEN 1
            ELSE 0
        END AS went_dark

    FROM read_parquet('{MAR}')

    WHERE gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
    """
).df()

print(
    "March outcome rows:",
    len(MAR_OUTCOME)
)


# ============================================================
# 4. CREATE MODELING DATASET
# ============================================================

model_df = FEB_FEATURES.merge(
    MAR_OUTCOME,
    on=[
        "client_hash_id",
        "content_hash_id"
    ],
    how="inner"
)

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "content_age_days"
]

model_df = model_df.dropna(
    subset=feature_cols + [
        "went_dark",
        "client_hash_id"
]
).copy()

print(
    "Final modeling rows:",
    len(model_df)
)


# ============================================================
# 5. GROUPED TRAIN/TEST SPLIT
# ============================================================

X = model_df[feature_cols]
y = model_df["went_dark"]
groups = model_df["client_hash_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        X,
        y,
        groups=groups
    )
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print(
    "Training rows:",
    len(X_train)
)

print(
    "Test rows:",
    len(X_test)
)


# ============================================================
# 6. TRAIN LOGISTIC REGRESSION MODEL
# ============================================================

model = Pipeline(
    [
        (
            "scaler",
            StandardScaler()
        ),

        (
            "logistic_regression",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

model.fit(
    X_train,
    y_train
)

model_scores = model.predict_proba(
    X_test
)[:, 1]

print("Logistic Regression: TRAINED")


# ============================================================
# 7. CREATE TEST-SET ACTION QUEUE
# ============================================================

action_queue = model_df.iloc[
    test_idx
].copy()

action_queue[
    "model_score"
] = model_scores


# ============================================================
# 8. RECREATE WEEK-4 BASELINE SCORE
# ============================================================

action_queue[
    "baseline_score"
] = np.select(
    [
        (
            (action_queue["content_age_days"] >= 365)
            &
            (action_queue["gsc_impressions"] < 100)
        ),

        (
            (action_queue["content_age_days"] >= 365)
            &
            (action_queue["gsc_impressions"] < 1000)
        ),

        (
            action_queue["content_age_days"].between(
                180,
                364
            )
            &
            (action_queue["gsc_impressions"] < 100)
        ),

        (
            action_queue["content_age_days"].between(
                180,
                364
            )
            &
            (action_queue["gsc_impressions"] < 1000)
        ),

        action_queue["gsc_impressions"] < 100,

        action_queue["gsc_impressions"] < 1000
    ],

    [
        4,
        3,
        3,
        2,
        2,
        1
    ],

    default=0
)


# ============================================================
# 9. ASSIGN REASON CODES
# ============================================================

action_queue[
    "reason_code"
] = np.select(
    [
        (
            (action_queue["content_age_days"] >= 365)
            &
            (action_queue["gsc_impressions"] < 100)
        ),

        (
            (action_queue["content_age_days"] >= 365)
            &
            (action_queue["gsc_impressions"] < 1000)
        ),

        (
            action_queue["content_age_days"].between(
                180,
                364
            )
            &
            (action_queue["gsc_impressions"] < 100)
        ),

        (
            action_queue["content_age_days"].between(
                180,
                364
            )
            &
            (action_queue["gsc_impressions"] < 1000)
        ),

        action_queue["gsc_impressions"] < 100,

        action_queue["gsc_impressions"] < 1000
    ],

    [
        "stale_low_visibility",
        "stale_moderate_visibility",
        "aging_low_visibility",
        "aging_moderate_visibility",
        "low_visibility",
        "moderate_visibility"
    ],

    default="monitor"
)


# ============================================================
# 10. ASSIGN HUMAN REVIEW ACTION
# ============================================================

action_queue[
    "action"
] = np.select(
    [
        action_queue["reason_code"].isin(
            [
                "stale_low_visibility",
                "stale_moderate_visibility",
                "aging_low_visibility",
                "aging_moderate_visibility"
            ]
        ),

        action_queue["reason_code"].isin(
            [
                "low_visibility",
                "moderate_visibility"
            ]
        )
    ],

    [
        "review_refresh",
        "review_visibility"
    ],

    default="monitor"
)


# ============================================================
# 11. RANK PAGES
# ============================================================

action_queue = action_queue.sort_values(
    [
        "model_score",
        "baseline_score"
    ],
    ascending=[
        False,
        False
    ]
).reset_index(
    drop=True
)

action_queue.insert(
    0,
    "rank",
    np.arange(
        1,
        len(action_queue) + 1
    )
)


# ============================================================
# 12. DISPLAY TOP 20
# ============================================================

display(
    action_queue[
        [
            "rank",
            "model_score",
            "baseline_score",
            "content_age_days",
            "gsc_impressions",
            "gsc_clicks",
            "reason_code",
            "action"
        ]
    ].head(20)
)

print(
    "Ranked action queue: READY"
)

DuckDB connection: READY
FlyRank warehouse: READY


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

February feature rows: 153559


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March outcome rows: 176738
Final modeling rows: 134238
Training rows: 88344
Test rows: 45894
Logistic Regression: TRAINED


,rank,model_score,baseline_score,content_age_days,gsc_impressions,gsc_clicks,reason_code,action
0,1,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh
1,2,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh
2,3,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh
3,4,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh
4,5,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh
5,6,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh
6,7,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh
7,8,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh
8,9,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh
9,10,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh


Ranked action queue: READY


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended Use

The action playbook is intended for a content or SEO team that needs to prioritize limited review capacity.

A reviewer can use the ranking to decide which pages to inspect first and then combine the model signal with page context, search intent, business importance, seasonality, and editorial judgment.

### Limits

The model score is not an instruction to automatically refresh, remove, or rewrite a page.

The observed `went_dark` outcome is a proxy based on March GSC clicks and does not measure whether a refresh would succeed.

The ranking is based on the available features and evaluation population, so its performance may differ for other clients, time periods, or future data.

In [31]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Summary of intended actions

action_summary = (
    action_queue
    .groupby("action")
    .agg(
        pages=("rank", "size"),
        average_model_score=("model_score", "mean")
    )
    .reset_index()
)

action_summary["average_model_score"] = (
    action_summary["average_model_score"].round(4)
)

display(action_summary)

print("Action categories:", action_queue["action"].nunique())


,action,pages,average_model_score
0,monitor,9111,0.0755
1,review_refresh,22545,0.7238
2,review_visibility,14238,0.6989


Action categories: 3


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human Review Requirements

Before taking any content action, a human reviewer should check:

- Whether the page still matches the intended search intent.
- Whether the topic is still relevant to the audience.
- Whether the content is accurate and up to date.
- Whether the page has seasonal or temporary demand patterns.
- Whether the page has important business or editorial context.
- Whether the observed search signals are sufficient to justify review.
- Whether the proposed action is appropriate after inspecting the actual page.

### No-Go List

The following decisions should not be automated from this ranking alone:

- Automatically deleting a page.
- Automatically rewriting or publishing content.
- Automatically changing search strategy.
- Automatically declaring a page unsuccessful.
- Automatically treating a high score as proof that a refresh will improve performance.
- Automatically making business or editorial decisions without human review.

In [32]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Human-review gate checklist

review_gate = pd.DataFrame({
    "check": [
        "Search intent reviewed",
        "Content relevance reviewed",
        "Accuracy/freshness reviewed",
        "Seasonality considered",
        "Business/editorial context reviewed",
        "Model signal treated as decision support"
    ],
    "required_before_action": [True] * 6
})

display(review_gate)

assert review_gate["required_before_action"].all()

print("Human review gate: DEFINED")


,check,required_before_action
0,Search intent reviewed,True
1,Content relevance reviewed,True
2,Accuracy/freshness reviewed,True
3,Seasonality considered,True
4,Business/editorial context reviewed,True
5,Model signal treated as decision support,True


Human review gate: DEFINED


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring and Retraining Triggers

The recommendations should be reviewed if the underlying data or measured model performance changes.

Potential triggers include:

- A sustained change in the distribution of impressions, clicks, or content age.
- A measurable decline in Precision@50 or Average Precision on a later evaluation window.
- A change in the relationship between the model ranking and the observed outcome.
- A change in the available data schema or feature definitions.
- A new period or client population where the current validation results no longer represent the intended use.

A later evaluation should use a new time window and should keep future outcomes separate from decision-time features.

The model should be retrained or revalidated when monitoring shows that its measured ranking quality is no longer adequate for the intended decision-support use.

In [33]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Monitoring thresholds are documented as review triggers,
# not as automatic retraining commands.

monitoring_triggers = pd.DataFrame({
    "trigger": [
        "Feature distribution changes materially",
        "Precision@50 declines on a later evaluation window",
        "Average Precision declines on a later evaluation window",
        "Feature definitions or schema change",
        "New client/time population differs from validation population"
    ],
    "response": [
        "Review and revalidate",
        "Revalidate model",
        "Revalidate model",
        "Audit feature pipeline",
        "Run a new validation"
    ]
})

display(monitoring_triggers)

print("Monitoring/retraining playbook: READY")


,trigger,response
0,Feature distribution changes materially,Review and revalidate
1,Precision@50 declines on a later evaluation wi...,Revalidate model
2,Average Precision declines on a later evaluati...,Revalidate model
3,Feature definitions or schema change,Audit feature pipeline
4,New client/time population differs from valida...,Run a new validation


Monitoring/retraining playbook: READY


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Public-Safe Paper Exports

The action queue is exported to `work/outputs/` so that the research paper can reuse the analysis output.

The public-facing recommendation table uses recommendation IDs rather than client or content identifiers.

The exported artifacts contain model and signal information needed to explain the ranking while avoiding private client information.

In [34]:
# ============================================================
# W07 — Build Ranked Action Queue
# ============================================================

import os
import pandas as pd
import numpy as np
import duckdb
from google.colab import userdata

# ------------------------------------------------------------
# 1. Connect to the FlyRank warehouse
# ------------------------------------------------------------

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf_secret
    (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}')
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FEB = f"{REL}/fact_content_daily_performance/month=2026-02/*.parquet"
MAR = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"
DIM_CONTENT = f"{REL}/dim_content.parquet"

print("DuckDB connection: READY")
print("Warehouse sources: READY")


# ------------------------------------------------------------
# 2. Build February decision-time features
# ------------------------------------------------------------

FEB_FEATURES = con.sql(f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks
    FROM read_parquet('{FEB}')
    WHERE gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
),

content AS (
    SELECT
        client_hash_id,
        content_hash_id,
        content_created_date
    FROM read_parquet('{DIM_CONTENT}')
)

SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.gsc_impressions,
    f.gsc_clicks,

    DATE_DIFF(
        'day',
        c.content_created_date,
        DATE '2026-02-28'
    ) AS content_age_days

FROM feb f

JOIN content c
    ON f.client_hash_id = c.client_hash_id
   AND f.content_hash_id = c.content_hash_id

WHERE c.content_created_date IS NOT NULL
""").df()

print("February feature rows:", len(FEB_FEATURES))


# ------------------------------------------------------------
# 3. Build March observed outcome
# ------------------------------------------------------------

MAR_OUTCOME = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    CASE
        WHEN SUM(gsc_clicks) = 0 THEN 1
        ELSE 0
    END AS went_dark

FROM read_parquet('{MAR}')

WHERE gsc_data_available IS TRUE

GROUP BY
    client_hash_id,
    content_hash_id
""").df()

print("March outcome rows:", len(MAR_OUTCOME))


# ------------------------------------------------------------
# 4. Combine February features + March outcome
# ------------------------------------------------------------

model_df = FEB_FEATURES.merge(
    MAR_OUTCOME,
    on=[
        "client_hash_id",
        "content_hash_id"
    ],
    how="inner"
)

model_df = model_df.dropna(
    subset=[
        "gsc_impressions",
        "gsc_clicks",
        "content_age_days",
        "went_dark"
    ]
).copy()

print("Modeling rows:", len(model_df))


# ------------------------------------------------------------
# 5. Train Logistic Regression
# ------------------------------------------------------------

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "content_age_days"
]

X = model_df[feature_cols]
y = model_df["went_dark"]
groups = model_df["client_hash_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        X,
        y,
        groups=groups
    )
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

model = Pipeline([
    ("scaler", StandardScaler()),

    ("logistic_regression", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

model.fit(
    X_train,
    y_train
)

model_scores = model.predict_proba(
    X_test
)[:, 1]

print("Model training: COMPLETE")
print("Test rows:", len(X_test))


# ------------------------------------------------------------
# 6. Create ranked action queue
# ------------------------------------------------------------

action_queue = model_df.iloc[test_idx].copy()

action_queue["model_score"] = model_scores


# ------------------------------------------------------------
# 7. Recreate Week-4 baseline score
# ------------------------------------------------------------

action_queue["baseline_score"] = np.select(
    [
        (
            (action_queue["content_age_days"] >= 365) &
            (action_queue["gsc_impressions"] < 100)
        ),

        (
            (action_queue["content_age_days"] >= 365) &
            (action_queue["gsc_impressions"] < 1000)
        ),

        (
            (action_queue["content_age_days"].between(180, 364)) &
            (action_queue["gsc_impressions"] < 100)
        ),

        (
            (action_queue["content_age_days"].between(180, 364)) &
            (action_queue["gsc_impressions"] < 1000)
        ),

        action_queue["gsc_impressions"] < 100,

        action_queue["gsc_impressions"] < 1000
    ],

    [
        4,
        3,
        3,
        2,
        2,
        1
    ],

    default=0
)


# ------------------------------------------------------------
# 8. Assign reason codes
# ------------------------------------------------------------

action_queue["reason_code"] = np.select(
    [
        (
            (action_queue["content_age_days"] >= 365) &
            (action_queue["gsc_impressions"] < 100)
        ),

        (
            (action_queue["content_age_days"] >= 365) &
            (action_queue["gsc_impressions"] < 1000)
        ),

        (
            (action_queue["content_age_days"].between(180, 364)) &
            (action_queue["gsc_impressions"] < 100)
        ),

        (
            (action_queue["content_age_days"].between(180, 364)) &
            (action_queue["gsc_impressions"] < 1000)
        ),

        action_queue["gsc_impressions"] < 100,

        action_queue["gsc_impressions"] < 1000
    ],

    [
        "stale_low_visibility",
        "stale_moderate_visibility",
        "aging_low_visibility",
        "aging_moderate_visibility",
        "low_visibility",
        "moderate_visibility"
    ],

    default="monitor"
)


# ------------------------------------------------------------
# 9. Assign actions
# ------------------------------------------------------------

action_queue["action"] = np.select(
    [
        action_queue["reason_code"].isin([
            "stale_low_visibility",
            "stale_moderate_visibility",
            "aging_low_visibility",
            "aging_moderate_visibility"
        ]),

        action_queue["reason_code"].isin([
            "low_visibility",
            "moderate_visibility"
        ])
    ],

    [
        "review_refresh",
        "review_visibility"
    ],

    default="monitor"
)


# ------------------------------------------------------------
# 10. Rank the queue
# ------------------------------------------------------------

action_queue = action_queue.sort_values(
    [
        "model_score",
        "baseline_score"
    ],
    ascending=[
        False,
        False
    ]
).reset_index(drop=True)

action_queue.insert(
    0,
    "rank",
    range(
        1,
        len(action_queue) + 1
    )
)

print("Action queue rows:", len(action_queue))

display(
    action_queue[
        [
            "rank",
            "model_score",
            "baseline_score",
            "content_age_days",
            "gsc_impressions",
            "gsc_clicks",
            "reason_code",
            "action"
        ]
    ].head(20)
)

print("Ranked Action Queue: READY")

DuckDB connection: READY
Warehouse sources: READY


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

February feature rows: 153559


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March outcome rows: 176738
Modeling rows: 134238
Model training: COMPLETE
Test rows: 45894
Action queue rows: 45894


,rank,model_score,baseline_score,content_age_days,gsc_impressions,gsc_clicks,reason_code,action
0,1,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh
1,2,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh
2,3,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh
3,4,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh
4,5,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh
5,6,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh
6,7,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh
7,8,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh
8,9,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh
9,10,0.823752,4,416,1.0,0.0,stale_low_visibility,review_refresh


Ranked Action Queue: READY


In [35]:
# ============================================================
# W07 — Export Public-Safe Queue for Research Paper
# ============================================================

import os

os.makedirs("work/outputs", exist_ok=True)

paper_queue = action_queue[
    [
        "model_score",
        "baseline_score",
        "content_age_days",
        "gsc_impressions",
        "gsc_clicks",
        "reason_code",
        "action"
    ]
].head(20).copy()

paper_queue.insert(
    0,
    "recommendation_id",
    [
        f"R{i:02d}"
        for i in range(
            1,
            len(paper_queue) + 1
        )
    ]
)

paper_queue["model_score"] = (
    paper_queue["model_score"].round(4)
)

queue_path = (
    "work/outputs/action_playbook_queue.csv"
)

paper_queue.to_csv(
    queue_path,
    index=False
)

print("Exported:", queue_path)
print("Rows exported:", len(paper_queue))

display(paper_queue)

print("Paper export: READY")

Exported: work/outputs/action_playbook_queue.csv
Rows exported: 20


,recommendation_id,model_score,baseline_score,content_age_days,gsc_impressions,gsc_clicks,reason_code,action
0,R01,0.8238,4,416,1.0,0.0,stale_low_visibility,review_refresh
1,R02,0.8238,4,416,1.0,0.0,stale_low_visibility,review_refresh
2,R03,0.8238,4,416,1.0,0.0,stale_low_visibility,review_refresh
3,R04,0.8238,4,416,1.0,0.0,stale_low_visibility,review_refresh
4,R05,0.8238,4,416,1.0,0.0,stale_low_visibility,review_refresh
5,R06,0.8238,4,416,1.0,0.0,stale_low_visibility,review_refresh
6,R07,0.8238,4,416,1.0,0.0,stale_low_visibility,review_refresh
7,R08,0.8238,4,416,1.0,0.0,stale_low_visibility,review_refresh
8,R09,0.8238,4,416,1.0,0.0,stale_low_visibility,review_refresh
9,R10,0.8238,4,416,1.0,0.0,stale_low_visibility,review_refresh


Paper export: READY


In [36]:
# ============================================================
# W07 — Public-Safe Export Verification
# ============================================================

public_safe_forbidden = [
    "client_hash_id",
    "content_hash_id",
    "client_name",
    "domain",
    "url",
    "query"
]

present_forbidden = [
    col
    for col in paper_queue.columns
    if col.lower() in public_safe_forbidden
]

print(
    "Forbidden public fields found:",
    present_forbidden
)

assert present_forbidden == []

assert "recommendation_id" in paper_queue.columns

assert os.path.exists(queue_path)

assert len(paper_queue) <= 20

print("Public-safe export check: PASSED")
print("Paper export: READY")

Forbidden public fields found: []
Public-safe export check: PASSED
Paper export: READY


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.